In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, LogLocator
import copy
import xarray as xr
import numpy as np
import geopandas as gpd
import rasterio as rio
from datetime import datetime
import matplotlib.colors as colors
from pathlib import Path
from matplotlib.patches import Rectangle
from tqdm.notebook import tqdm
from idd_forecast_mbp import constants as rfc
import matplotlib.gridspec as gridspec
from idd_forecast_mbp.helper_functions import read_income_paths
from idd_forecast_mbp.parquet_functions import read_parquet_with_integer_ids
from idd_forecast_mbp.xarray_functions import read_netcdf_with_integer_ids, convert_to_xarray, write_netcdf

In [3]:
ssp_scenario_map = rfc.ssp_scenario_map
cause_map = rfc.cause_map

# Admin shapefile path
ADMIN_SHAPEFILE_TEMPLATE = "/snfs1/WORK/11_geospatial/admin_shapefiles/2023_10_30/lbd_standard_admin_{admin_num}.shp"

# MODELING_DATA_PATH 
PROCESSED_DATA_PATH = rfc.PROCESSED_DATA_PATH
MODELING_DATA_PATH = rfc.MODELING_DATA_PATH
FORECASTING_DATA_PATH = rfc.FORECASTING_DATA_PATH
VISUALIZATION_PATH = rfc.VISUALIZATION_PATH
UPLOAD_DATA_PATH = rfc.UPLOAD_DATA_PATH
FIGURES_PATH = rfc.FIGURES_PATH

VARIABLE_DATA_PATH = PROCESSED_DATA_PATH / "lsae_1209"

# Run date
run_date = "2025_08_13"

# Hierarchy path
hierarchy_df_path = f'{PROCESSED_DATA_PATH}/full_hierarchy_lsae_1209.parquet'
hierarchy_df = read_parquet_with_integer_ids(hierarchy_df_path)

In [4]:
best_run_date = "2025_08_11"


full_ds_path_template = "{UPLOAD_DATA_PATH}/upload_folders/{best_run_date}/full_aa_ds_{dah_scenario}.nc"
suit_ds_path = f"{PROCESSED_DATA_PATH}/suitability_data.nc"
income_ds_path = f"{PROCESSED_DATA_PATH}/income_data.nc"

In [5]:
dah_scenario = 'Baseline'
full_ds_path = full_ds_path_template.format(
    UPLOAD_DATA_PATH=UPLOAD_DATA_PATH,
    best_run_date=best_run_date,
    dah_scenario=dah_scenario
)
full_ds = read_netcdf_with_integer_ids(full_ds_path)

suit_ds = read_netcdf_with_integer_ids(suit_ds_path)
income_ds = read_netcdf_with_integer_ids(income_ds_path)

In [6]:
import pickle
bins_dictionary_path = '/ihme/homes/bcreiner/repos/idd-forecast-mbp/notebooks/09_figures/bins_dictionary.pkl'
with open(bins_dictionary_path, 'rb') as f:
    bins_dictionary = pickle.load(f)

In [33]:
def smart_format(val):
    # If integer, format as int. If <1, keep up to 2 decimals, else keep up to 2 decimals but drop trailing zeros.
    if float(val).is_integer():
        return f"{int(val)}"
    else:
        s = f"{val:.2f}".rstrip('0').rstrip('.')
        return s
def pretty_bin_labels(bins, le = False, ge = False, zero_bin = False):
    # bins: array-like of bin edges
    # fmt: format for numbers (default: 2 significant digits)

    if zero_bin:
        zero_ix = np.where(np.atleast_1d(bins) == 0)[0]
    labels = []
    for i in range(len(bins) - 1):
        left = smart_format(bins[i])
        right = smart_format(bins[i+1])
        if zero_bin and i == zero_ix:
            labels.append('0')
        elif zero_bin and i == zero_ix + 1:
            labels.append(f"0 - {right}")
        elif left == right:
            labels.append(left)
        else:
            labels.append(f"{left}–{right}")
    if le:
        labels[0] = f"< {smart_format(bins[1])}"
    if ge:
        labels[-1] = f"> {smart_format(bins[-2])}"
    

    return labels

def get_colors(n_bins, cmap_name='Reds'):
    cmap = plt.get_cmap(cmap_name)
    return [cmap(i / (n_bins - 1)) for i in range(n_bins)]

In [8]:
admin0_polygons = gpd.read_file(ADMIN_SHAPEFILE_TEMPLATE.format(admin_num=0))
admin0_polygons = admin0_polygons.rename(columns={"loc_id": "location_id"})
admin1_polygons = gpd.read_file(ADMIN_SHAPEFILE_TEMPLATE.format(admin_num=1))
admin1_polygons = admin1_polygons.rename(columns={"loc_id": "location_id"})
admin2_polygons = gpd.read_file(ADMIN_SHAPEFILE_TEMPLATE.format(admin_num=2))
admin2_polygons = admin2_polygons.rename(columns={"loc_id": "location_id"})

admin1_polygons = admin1_polygons.merge(admin0_polygons[['ADM0_CODE', 'location_id']].rename(columns={'location_id': 'A0_location_id'}),
                                        on='ADM0_CODE', how='left')
admin2_polygons = admin2_polygons.merge(admin0_polygons[['ADM0_CODE', 'location_id']].rename(columns={'location_id': 'A0_location_id'}),
                                        on='ADM0_CODE', how='left')

In [9]:
def get_threshold_locations(cause='malaria', threshold=0, metric='count', measure=None):
    """
    Get a list of countries where the cause is endemic based on the specified threshold.
    """
    if measure is not None:
        data_ds = full_ds.sel(cause=cause, year_id = 2022, measure = measure)
    else:
        data_ds = full_ds.sel(cause=cause, year_id = 2022)
    plot_data = data_ds.to_dataframe().reset_index()
    if threshold is not None:
        threshold_loc_ids = plot_data[plot_data[metric] > threshold]['location_id'].unique()
    else:
        threshold_loc_ids = plot_data['location_id'].unique()

    level_5_loc_ids = hierarchy_df[hierarchy_df['level'] == 5]['location_id'].values
    a2_loc_ids = threshold_loc_ids[np.isin(threshold_loc_ids, level_5_loc_ids)]
    a0_loc_ids = hierarchy_df[hierarchy_df['location_id'].isin(a2_loc_ids)]['A0_location_id'].values
    a1_loc_ids = hierarchy_df[hierarchy_df['parent_id'].isin(a0_loc_ids)]['location_id'].values
    a2_loc_ids = hierarchy_df[hierarchy_df['parent_id'].isin(a1_loc_ids)]['location_id'].values

    return a0_loc_ids, a1_loc_ids, a2_loc_ids

endemic_loc_id_list = {}
# Populate the dictionaries
for cause in ['malaria', 'dengue']:
    a0_loc_ids, a1_loc_ids, a2_loc_ids = get_threshold_locations(cause=cause)
    endemic_loc_id_list[cause] = {
        'a0_loc_ids': a0_loc_ids,
        'a1_loc_ids': a1_loc_ids,
        'a2_loc_ids': a2_loc_ids
    }

all_loc_id_list = {}
# Populate the dictionaries
for cause in ['malaria', 'dengue']:
    a0_loc_ids, a1_loc_ids, a2_loc_ids = get_threshold_locations(cause=cause, threshold=None)
    all_loc_id_list[cause] = {
        'a0_loc_ids': a0_loc_ids,
        'a1_loc_ids': a1_loc_ids,
        'a2_loc_ids': a2_loc_ids
    }

location_lists = {
    'endemic': {
        'loc_id_list': endemic_loc_id_list
    },
    'all': {
        'loc_id_list': all_loc_id_list
    }        
}

In [10]:
def get_bin_info(plot_dict, valid_data, map_data_masked):
    map_type = plot_dict['map_type']
    # Categorization logic (same as before)
    if plot_dict.get('custom_bins', None) is not None:
        bins = np.array(plot_dict['custom_bins'])
        n_bins = len(bins) - 1
        base_colormap = plt.colormaps[plot_dict['colors_dict']['base_cmap']]
        white_rgba = to_rgba('white')
        bin_colors = np.vstack([white_rgba, base_colormap(np.linspace(0.1, 0.9, n_bins - 1))])
        cmap = ListedColormap(bin_colors)
    elif map_type == 'outcome':
        max_value = np.nanmax(valid_data)
        bins = np.linspace(0, max_value, plot_dict['num_categories'] + 1)[1:]
        n_bins = len(bins) - 1
        base_colormap = plt.colormaps[plot_dict['colors_dict']['base_cmap']]
        white_rgba = to_rgba('white')
        bin_colors = np.vstack([white_rgba, base_colormap(np.linspace(0.1, 0.9, n_bins - 1))])
        cmap = ListedColormap(bin_colors)
    else:
        max_abs_value = max(abs(np.nanmin(valid_data)), abs(np.nanmax(valid_data)))
        half_categories = plot_dict['num_categories'] // 2
        positive_bounds = np.linspace(0, max_abs_value, half_categories + 1)[1:]
        negative_bounds = -positive_bounds[::-1]
        bins = np.concatenate([negative_bounds, [0], positive_bounds])
        n_bins = len(bins) - 1
        base_colormap = plt.colormaps[plot_dict['colors_dict']['base_cmap']]
        bin_colors = base_colormap(np.linspace(0.1, 0.9, n_bins))
        cmap = ListedColormap(bin_colors)

    categorical_data = np.full_like(map_data_masked, np.nan)
    for i in range(n_bins):
        if i == 0:
            mask = map_data_masked <= bins[i+1]
        elif i == n_bins - 1:
            mask = map_data_masked > bins[i]
        else:
            mask = (map_data_masked > bins[i]) & (map_data_masked <= bins[i+1])
        categorical_data[mask] = i

    bin_dict = {
        'bins': bins,
        'n_bins': n_bins,
        'bin_colors': bin_colors,
        'cmap': cmap,
        'categorical_data': categorical_data
    }
    plot_dict['bin_dict'] = bin_dict
    
    return plot_dict

In [79]:
def get_raster_data(plot_dict):
    measure = plot_dict['measure']
    resolution = plot_dict['resolution']

    plot_data = {}
    for period in periods:
        period_dict = plot_dict[period]
        start_year = period_dict['start_year']
        end_year = period_dict['end_year']

        reference_year = start_year
        reference_population, _ = load_population_data(reference_year, resolution=resolution)
        period_count = np.zeros_like(reference_population)
        period_rate = np.zeros_like(reference_population)
        period_population = np.zeros_like(reference_population)

        for year in range(start_year, end_year + 1):
            population_data, transform = load_population_data(year, resolution="0.1")
            period_population += population_data
            cov_dict = {'cov': measure, 'year': year, 'ssp_scenario': period_dict.get('ssp_scenario', 'ssp245')}
            cov_ds = load_cov_data(cov_dict)
            # Handle both flood and storm data by squeezing any extra dimensions
            cov_value = cov_ds['value'].squeeze()
            yearly_impact_per_capita = reproject_cov_slice(cov_value, population_data, transform)
            period_rate += yearly_impact_per_capita
            yearly_impact = yearly_impact_per_capita * population_data
            period_count += yearly_impact

        if statistic == 'mean':
            period_rate = period_rate / (end_year - start_year + 1)
            period_count = period_count / (end_year - start_year + 1)
            period_population = period_population / (end_year - start_year + 1)

        period_data = {
            'period_count': period_count,
            'period_rate': period_rate
        }
        plot_data[period] = period_data

    
    map_type = plot_dict['map_type']
    metric = 'period_impact_per_capita' if plot_dict['per_capita'] else 'period_impact'
    if map_type == 'outcome':
        map_data = plot_data['period_1'][metric]
    else:
        map_data = plot_data['period_2'][metric] - plot_data['period_1'][metric]

    map_data_masked = np.copy(map_data)
    map_data_masked[plot_data['combined_water_mask']] = np.nan
    valid_data = map_data_masked[~np.isnan(map_data_masked)]

    plot_dict = get_bin_info(plot_dict, valid_data, map_data_masked)


    return plot_dict

In [49]:
def get_outcome_data(plot_dict):
    map_type = plot_dict['map_type']
    period_1 = plot_dict['period_1']
    if map_type != 'outcome':
        period_2 = plot_dict['period_2']

    outcome_dfs = []
    for input_data in [period_1, period_2]:
        if input_data is not None:
            scenario, year = input_data
            if plot_dict['outcome_type'] == 'suitability':
                outcome_ds = suit_ds.sel(cause=plot_dict['cause'], location_id=plot_dict['map_a2_loc_ids'], year_id=year, 
                                         ssp_scenario=scenario)
                outcome_df = outcome_ds.to_dataframe().reset_index()
                outcome_df = outcome_df.rename(columns={'suitability': 'val'})
            else:
                metric = plot_dict['metric']
                outcome_ds = full_ds.sel(cause=plot_dict['cause'], measure=plot_dict['measure'],  
                                         ssp_scenario=scenario, year_id=year, location_id=plot_dict['map_a2_loc_ids'])
                outcome_df = outcome_ds.to_dataframe().reset_index()[['location_id', 'year_id', 'ssp_scenario', metric]].rename(columns={metric: 'val'})
                if metric == 'rate':
                    outcome_df['val'] = outcome_df['val'] * 100000
            # If A0_location_id isn't in the outcome_df, merge it from hierarchy_df
            if 'A0_location_id' not in outcome_df.columns:
                outcome_df = outcome_df.merge(hierarchy_df[['location_id', 'A0_location_id']], on='location_id', how='left')
            outcome_dfs.append(outcome_df)
        else:
            continue
    df = pd.concat(outcome_dfs, ignore_index=True)

    if map_type == 'scenario_comparison':
        df = get_scenario_difference(df, plot_dict)
    elif map_type == 'change':
        df = get_outcome_change(df, plot_dict)
    else:
        df = df[['location_id', 'val']]

    return df

def get_outcome_change(df, plot_dict):
    """Calculate suitability change between two years."""
    start_data = df[df['year_id'] == plot_dict['start_year']][['location_id', 'val']]
    end_data = df[df['year_id'] == plot_dict['end_year']][['location_id', 'val']]
    
    change_df= pd.merge(start_data, end_data, on='location_id', suffixes=('_start', '_end'))
    change_df['change'] = change_df['val_end'] - change_df['val_start']
    change_df['relative_change'] = change_df['change'] / change_df['val_start'] * 100 if change_df['val_start'].any() else np.nan
    
    return change_df[['location_id', 'change', 'relative_change']]

def get_scenario_difference(df, plot_dict):
    """Calculate difference between two scenarios for the same year."""
    base_df = df[df['ssp_scenario'] == plot_dict['base_scenario']][['location_id', 'val']]
    comparison_df = df[df['ssp_scenario'] == plot_dict['comparison_scenario']][['location_id', 'val']]

    difference_df = pd.merge(base_df, comparison_df, on='location_id', 
                     suffixes=('_base', '_comparison'))
    difference_df['diff'] = difference_df['val_comparison'] - difference_df['val_base']
    difference_df['relative_diff'] = difference_df['diff'] / difference_df['val_base'] * 100 if difference_df['val_base'].any() else np.nan
    
    return difference_df[['location_id', 'diff', 'relative_diff']]


def create_outcome_colormap(bins, cmap_name = 'Reds'):
    """Create colormap and bins for suitability values (0-365 days)."""
    if cmap_name is None:
        cmap_name = 'Reds'
    n_bins = len(bins) - 1
    bin_labels = pretty_bin_labels(bins, le=False, ge=True)
    bin_colors = get_colors(len(bins) - 1, cmap_name=cmap_name)

    cmap = ListedColormap(bin_colors)
    norm = BoundaryNorm(bins, cmap.N, clip = True)
    
    return bin_colors, bin_labels, cmap, norm

def create_diverging_colors(n_bins, cmap_name='RdBu_r'):
    """
    Create a diverging color palette by generating n_bins colors
    
    Parameters:
    - n_bins: int, desired number of final colors
    - cmap_name: str, name of the matplotlib colormap to use
    
    Returns:
    - list of color tuples for the final colormap
    """
    if cmap_name is None:
        cmap_name = 'RdBu_r'
    # Get the full colormap
    cmap_full = plt.get_cmap(cmap_name)
    
    # Create n_bins colors
    bin_colors = [cmap_full(i / (n_bins - 1)) for i in range(n_bins)]
        
    return bin_colors

def create_change_colormap(bins, cmap_name = 'RdBu_r', remove_middle = False, force_white=False):
    """Create colormap and bins for change/difference values."""
    if cmap_name is None:
        cmap_name = 'RdBu_r'
    n_bins = len(bins) - 1
    if remove_middle:
        bin_labels = pretty_bin_labels(bins, le=True, ge=True)
        bin_colors = create_diverging_colors(n_bins + 2, cmap_name=cmap_name)
        mid_index = (n_bins + 2) // 2
        if force_white:
            # there are an odd number of colors. We want to set the middle one to white and delete the colors above and below it
            bin_colors = bin_colors[:(mid_index - 1)] + ["#ffffff"] + bin_colors[(mid_index+1):]
            # print("Colors after edit:", bin_colors)
        else:
            bin_colors = bin_colors[:(mid_index - 1)] + bin_colors[mid_index+1:]
    else:
        bin_labels = pretty_bin_labels(bins, le=True, ge=True)
        bin_colors = create_diverging_colors(n_bins, cmap_name=cmap_name)
        if force_white:
            # there are an odd number of colors. We want to set the middle one to white and delete the colors above and below it
            mid_index = n_bins // 2
            bin_colors[mid_index] = '#ffffff'

    cmap = ListedColormap(bin_colors)
    norm = BoundaryNorm(bins, cmap.N, clip = True)
    
    return bin_colors, bin_labels, cmap, norm

def setup_map_plot(ax_map, plot_dict):
    map_dict = plot_dict['map_dict']
    figure_dict = plot_dict['figure_dict']
    map_extent = map_dict.get('map_extent', [-180, 180, -90, 90])
    ax_map.set_extent(map_extent, crs=ccrs.PlateCarree())
    # Add geographic features
    ax_map.add_feature(cfeature.OCEAN, facecolor=figure_dict['water_color'], alpha=figure_dict['water_alpha'], zorder=0)
    ax_map.coastlines(linewidth=0.5)
    ax_map.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='gray')
    
    return ax_map

def plot_base_admins(ax_map, plot_dict):
    """Plot base polygon layers."""
    map_dict = plot_dict['map_dict']
    admin0_polygons = map_dict['admin0_polygons']
    admin1_polygons = map_dict['admin1_polygons']
    # Plot all admin2 areas in grey as background
    admin0_polygons.plot(ax=ax_map, color='lightgrey', edgecolor='black', 
                        linewidth=0, transform=ccrs.PlateCarree())
    if map_dict.get('plot_admin1s', False):
        if plot_dict['map_a1_loc_ids'] is not None:
            admin1s_to_plot = admin1_polygons[~admin1_polygons['location_id'].isin(plot_dict['map_a1_loc_ids'])]
        else:
            admin1s_to_plot = admin1_polygons
        admin1s_to_plot.boundary.plot(ax=ax_map, color='darkgrey', linewidth=0.25, 
                                            transform=ccrs.PlateCarree())
    
def plot_data_admins(ax_map, plot_dict, linewidth=0):
    """Plot polygons with data colors."""
    data_dict = plot_dict['data_dict']
    
    admin2_endemic = admin2_polygons[admin2_polygons['location_id'].isin(plot_dict['map_a2_loc_ids'])]
    admin2_with_data = admin2_endemic.merge(data_dict['plot_data'], on='location_id', how='left')
    admin2_with_data.plot(column=data_dict['data_column'], ax=ax_map, cmap=plot_dict['bin_dict']['cmap'], norm=plot_dict['bin_dict']['norm'], 
                         legend=False, edgecolor=None, linewidth=linewidth, 
                         transform=ccrs.PlateCarree())
    # Add boundaries

    admin0_polygons.boundary.plot(ax=ax_map, color='black', linewidth=0.5, 
                                 transform=ccrs.PlateCarree())


def add_colorbar(fig, ax, plot_dict):
    figure_dict = plot_dict['figure_dict']
    legend_dict = plot_dict['legend_dict']
    bins = plot_dict['bin_dict']['bins']
    bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]
    sm = plt.cm.ScalarMappable(cmap=plot_dict['bin_dict']['cmap'], norm=plot_dict['bin_dict']['norm'])
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', 
                        shrink=legend_dict['color_bar_dict']['shrink'], 
                        pad=legend_dict['color_bar_dict']['pad'],
                        aspect=legend_dict['color_bar_dict']['aspect'],
                        fraction=legend_dict['color_bar_dict']['fraction'],
                        ticks=bin_centers)
    cbar.set_ticklabels(figure_dict['bin_labels'], fontsize=figure_dict['tick_font_size'])
    cbar.set_label(figure_dict['colorbar_label'], fontsize=figure_dict['colorbar_title_font_size'])


In [51]:
def draw_legend_bins(ax, plot_dict):
    map_type = plot_dict['map_type']
    legend_dict = plot_dict['legend_dict']
    bin_dict = plot_dict['bin_dict']
    legend_panel = legend_dict['legend_panel']
    legend_bin_spacing = legend_panel['legend_bin_spacing']
    legend_margin = legend_panel['legend_margin']
    bin_colors = bin_dict['bin_colors']
    
    bin_bottom = legend_panel['bin_bottom']
    bin_top = legend_panel['bin_top']
    bin_label_gap = legend_panel['bin_label_gap']

    bin_height = bin_top - bin_bottom
    bin_label_y = bin_bottom - bin_label_gap

    if map_type == 'outcome':
        category_labels = pretty_bin_labels(bin_dict['bins'], zero_bin=True, le=False, ge=True)
    else:
        category_labels = pretty_bin_labels(bin_dict['bins'], le=True, ge=True)

    bin_dict['n_bins'] = len(category_labels)
    n_bins = bin_dict['n_bins']

    bin_width = (1 - 2 * legend_margin - (n_bins - 1) * legend_bin_spacing) / n_bins
    
    # bin_width = min(bin_width, 0.1)
    bin_left = np.arange(legend_margin, legend_margin + n_bins * (bin_width + legend_bin_spacing), bin_width + legend_bin_spacing)
    bin_center = bin_left + bin_width / 2
    bin_shift = 0.5 - bin_center.mean()
    bin_left += bin_shift
    # bin_left = np.arange(margin, margin + bin_dict['n_bins'] * (bin_width + legend_bin_spacing), bin_width + legend_bin_spacing)

    # print("bin_left:", bin_left)
    # Draw legend rectangles and labels
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    for i in range(n_bins):
        rect = Rectangle((bin_left[i], bin_bottom), bin_width, bin_height, facecolor=bin_colors[i], edgecolor='black', linewidth=0.5)
        ax.add_patch(rect)
        ax.text(bin_left[i] + bin_width / 2, bin_label_y, category_labels[i], ha='center', va='top',
                        fontsize=plot_dict['fontsizes']['legend_label_fontsize'])

In [39]:




def add_legend_panel_colorbar(fig, ax_legend, plot_dict):
    """
    Fixed version that uses the calculated parameters properly
    """
    ax_legend.clear()
    ax_legend.axis('off')

    # Get colorbar data
    bin_dict = plot_dict['bin_dict']
    cmap = bin_dict['cmap'] 
    norm = bin_dict['norm']
    bins = bin_dict.get('bins', None)
    bin_labels = bin_dict.get('bin_labels', None)
    colorbar_label = plot_dict['full_outcome_label']
    
    color_bar_dict = plot_dict['legend_dict']['color_bar_dict']

    # Create ScalarMappable
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])


    exact_fit = True
    legend_pos = ax_legend.get_position()
    # Use the ratios from calculate_colorbar_params
    colorbar_width_ratio = color_bar_dict.get('colorbar_width_ratio', 0.7)
    colorbar_height_ratio = color_bar_dict.get('colorbar_height_ratio', 0.3)

            
    # Calculate colorbar size and position
    cbar_width = legend_pos.width * colorbar_width_ratio
    cbar_height = legend_pos.height * colorbar_height_ratio
    
    cbar_left = legend_pos.x0 + cbar_width
    cbar_bottom = legend_pos.y0 + cbar_height

    cbar_ax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])

    extend_option = 'max' if plot_dict.get('extend_colorbar', False) else 'neither'
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend=extend_option)

    if bins is not None and bin_labels is not None:
        if hasattr(norm, 'boundaries'):
            # For BoundaryNorm, use the boundaries
            boundaries = list(norm.boundaries)
            # Remove infinite values for display
            display_boundaries = [b for b in boundaries if not np.isinf(b)]
            if len(display_boundaries) > len(bin_labels):
                display_boundaries = display_boundaries[:len(bin_labels)]
            
            cbar.set_ticks(display_boundaries)
            cbar.set_ticklabels(bin_labels[:len(display_boundaries)])
        else:
            # Fallback for other norm types
            tick_positions = np.linspace(norm.vmin, norm.vmax, len(bin_labels))
            cbar.set_ticks(tick_positions)
            cbar.set_ticklabels(bin_labels)
        
        # Style the colorbar
        cbar.set_label(colorbar_label, fontsize=10, labelpad=5)
        cbar.ax.tick_params(labelsize=9, rotation=0, pad=2)

    return cbar

def add_legend(fig, ax, plot_dict):
    if plot_dict['have_legend_panel']:
        if plot_dict['legend_dict']['use_colorbar']:
            add_legend_panel_colorbar(fig, ax, plot_dict)
        else:
            draw_legend_bins(ax, plot_dict)
    else:
        add_colorbar(fig, ax, plot_dict)

def add_inset(ax, figure_dict):
    """Add legend for non-endemic areas."""
    inset_elements = [Patch(facecolor='lightgrey', edgecolor='k', label=figure_dict['inset_label'])]
    ax.legend(handles=inset_elements, loc='lower left', bbox_to_anchor=(0.0, -0.02), 
             frameon=False, fontsize=figure_dict['inset_label_font_size'])



In [107]:
def get_admin2_data(plot_dict):

    cause = plot_dict['cause']
    map_type = plot_dict['map_type']
    measure = plot_dict['measure']
    metric = plot_dict['metric']
    outcome_type = plot_dict['outcome_type']
    outcome_label = plot_dict['outcome_label']
    full_outcome_label = plot_dict['full_outcome_label']
    bin_dict = plot_dict['bin_dict']
    bins = bin_dict['bins']

    if plot_dict['location_type'] == 'endemic':
        plot_dict['figure_dict']['inset_label'] = 'Non-endemic' if cause == 'malaria' else 'No local transmission'
        plot_dict['map_a0_loc_ids'] = location_lists['endemic']['loc_id_list'][cause]['a0_loc_ids']
        plot_dict['map_a1_loc_ids'] = location_lists['endemic']['loc_id_list'][cause]['a1_loc_ids']
        plot_dict['map_a2_loc_ids'] = location_lists['endemic']['loc_id_list'][cause]['a2_loc_ids']
    else:
        plot_dict['figure_dict']['inset_label'] = None
        plot_dict['map_a0_loc_ids'] = location_lists['all']['loc_id_list'][cause]['a0_loc_ids']
        plot_dict['map_a1_loc_ids'] = location_lists['all']['loc_id_list'][cause]['a1_loc_ids']
        plot_dict['map_a2_loc_ids'] = location_lists['all']['loc_id_list'][cause]['a2_loc_ids']
    
    outcome_dfs = []
    for period_num in range(1, len(plot_dict['periods']) + 1):
        period_dict = plot_dict[f'period_{period_num}']
        if plot_dict['outcome_type'] == 'suitability':
            outcome_ds = suit_ds.sel(cause=plot_dict['cause'], location_id=plot_dict['map_a2_loc_ids'], year_id=period_dict['start_year'], 
                                        ssp_scenario=period_dict['ssp_scenario'])
            outcome_df = outcome_ds.to_dataframe().reset_index()
            outcome_df = outcome_df.rename(columns={'suitability': 'val'})
        else:
            metric = plot_dict['metric']
            outcome_ds = full_ds.sel(cause=plot_dict['cause'], measure=plot_dict['measure'],  
                                        ssp_scenario=period_dict['ssp_scenario'], year_id=period_dict['start_year'], location_id=plot_dict['map_a2_loc_ids'])
            outcome_df = outcome_ds.to_dataframe().reset_index()[['location_id', 'year_id', 'ssp_scenario', metric]].rename(columns={metric: 'val'})
            if metric == 'rate':
                outcome_df['val'] = outcome_df['val'] * 100000
        outcome_dfs.append(outcome_df)
    if plot_dict['map_type'] == 'outcome':
        df = outcome_dfs[0][['location_id', 'val']].copy()
    else:
        df = outcome_dfs[0].merge(outcome_dfs[1], on='location_id', suffixes=('_period_1', '_period_2'))
        df['val'] = df['val_period_2'] - df['val_period_1']
        df = df[['location_id', 'val']].copy()

    if 'A0_location_id' not in df.columns:
        plot_data = df.merge(hierarchy_df[['location_id', 'A0_location_id']], on='location_id', how='left')
    
    # Prepare data and colormap based on map type
    data_column = 'val'
    if map_type == 'outcome':
        # Suitability map
        bin_colors, bin_labels, cmap, norm = create_outcome_colormap(bins = bins, cmap_name=plot_dict['colors_dict']['base_cmap'])
        title = f'{cause.capitalize()} {outcome_label} in {plot_dict['period_1']['start_year']} ({ssp_scenario_map[plot_dict['period_1']['ssp_scenario']]["name"]})'
        colorbar_label = f'{cause.capitalize()} {full_outcome_label}'
    elif map_type == 'change':
        # Change map
        bin_colors, bin_labels, cmap, norm = create_change_colormap(bins = bins, cmap_name=plot_dict['colors_dict']['base_cmap'], remove_middle=True, force_white=True)
        title = f'Change in {cause} {outcome_label}: {plot_dict['period_1']['start_year']} to {plot_dict['period_2']['start_year']} - {ssp_scenario_map[plot_dict['period_1']['ssp_scenario']]["name"]}'
        colorbar_label = f'Change in {cause.capitalize()} {full_outcome_label}'
    elif map_type == 'scenario_comparison':
        # Scenario comparison map
        bin_colors, bin_labels, cmap, norm = create_change_colormap(bins = bins, cmap_name=plot_dict['colors_dict']['base_cmap'], remove_middle=True, force_white=True)
        title = f'Difference in {cause} {outcome_label}: {ssp_scenario_map[plot_dict['period_2']['ssp_scenario']]["name"]} - {ssp_scenario_map[plot_dict['period_1']['ssp_scenario']]["name"]} ({plot_dict['period_1']['start_year']})'
        colorbar_label = f'Difference in {cause.capitalize()} {outcome_label} ({ssp_scenario_map[plot_dict['period_2']['ssp_scenario']]["name"]} - {ssp_scenario_map[plot_dict['period_1']['ssp_scenario']]["name"]})'

    data_dict = {
        'plot_data': plot_data,
        'data_column': data_column
    }
    plot_dict['data_dict'] = data_dict
    plot_dict['bin_dict']['n_bins'] = len(bins) - 1
    plot_dict['bin_dict']['bin_colors'] = bin_colors
    plot_dict['bin_dict']['bin_labels'] = bin_labels
    plot_dict['bin_dict']['cmap'] = cmap
    plot_dict['bin_dict']['norm'] = norm

    plot_dict['legend_dict']['legend_title'] = colorbar_label
    plot_dict['figure_dict']['title'] = title
    plot_dict['figure_dict']['colorbar_label'] = colorbar_label

    return plot_dict

In [108]:
def get_plot_data(plot_dict):


    data_type = plot_dict['data_type']
    if data_type == 'raster':
        plot_dict = get_raster_data(plot_dict)
    else:
        plot_dict = get_admin2_data(plot_dict)
       

    return plot_dict

In [109]:
def draw_red_rectangle(ax):
    from matplotlib.patches import Rectangle
    # Get axis position in figure coordinates
    pos = ax.get_position()
    rect = Rectangle((pos.x0, pos.y0), pos.width, pos.height,
                    linewidth=2, edgecolor='red', facecolor='none', zorder=100, transform=ax.figure.transFigure)
    ax.figure.patches.append(rect)

In [110]:
def calculate_colorbar_params(plot_dict):

    figure_dict = plot_dict['figure_dict']
    legend_dict = plot_dict['legend_dict']
    panel_ratios = figure_dict['panel_height_ratios']
    color_bar_dict = legend_dict['color_bar_dict']
    fig_width, fig_height = figure_dict['figsize']
    colorbar_width_ratio = color_bar_dict.get('colorbar_width_ratio', 0.7)
    colorbar_height_ratio = color_bar_dict.get('colorbar_height_ratio', 0.3)

    if plot_dict.get('legend_panel', False) and plot_dict['legend_dict'].get('use_colorbar', False):
        # Calculate actual dimensions for reference
        total_ratio = sum(panel_ratios) 
        legend_panel_height = fig_height * (panel_ratios[1] / total_ratio)
        
        # Store calculated dimensions (these will be used in add_legend_panel_colorbar)
        color_bar_dict['calculated_width'] = fig_width * colorbar_width_ratio
        color_bar_dict['calculated_height'] = legend_panel_height * colorbar_height_ratio
        color_bar_dict['legend_panel_height'] = legend_panel_height
    else:
        # For automatic positioning (not in legend panel)
        target_colorbar_width = fig_width * colorbar_width_ratio
        target_colorbar_height = fig_height * colorbar_height_ratio
        
        aspect = target_colorbar_width / target_colorbar_height
        shrink = colorbar_width_ratio
        
        color_bar_dict['aspect'] = max(8, min(40, aspect))
        color_bar_dict['shrink'] = max(0.2, min(0.95, shrink))  
        color_bar_dict['fraction'] = colorbar_height_ratio * 1.2

    return plot_dict

In [111]:
def calculate_colorbar_params(plot_dict, colorbar_width_ratio=0.7, colorbar_height_ratio=0.5):

    panel_height_ratios = plot_dict['figure_dict']['panel_height_ratios']
    figure_dict = plot_dict['figure_dict']
    legend_dict = plot_dict.get('legend_dict', {})    
    fig_width, fig_height = figure_dict['figsize']
    
    # Calculate actual panel dimensions
    total_ratio = sum(panel_height_ratios)
    legend_panel_height = fig_height * (panel_height_ratios[1] / total_ratio)
    
    # For horizontal colorbar in legend panel, we need different parameters
    if plot_dict.get('legend_panel', False):
        # Horizontal colorbar parameters
        target_colorbar_width = fig_width * colorbar_width_ratio
        target_colorbar_height = legend_panel_height * colorbar_height_ratio
        
        # For horizontal orientation, aspect ratio is width/height
        aspect = target_colorbar_width / target_colorbar_height
        
        # Store parameters for manual positioning (used in add_legend_panel_colorbar)
        legend_dict['color_bar_dict']['width_ratio'] = colorbar_width_ratio
        legend_dict['color_bar_dict']['height_ratio'] = colorbar_height_ratio
        legend_dict['color_bar_dict']['aspect'] = aspect
        legend_dict['color_bar_dict']['orientation'] = 'horizontal'
        
        # Also store absolute dimensions for reference
        legend_dict['color_bar_dict']['target_width'] = target_colorbar_width
        legend_dict['color_bar_dict']['target_height'] = target_colorbar_height
        legend_dict['color_bar_dict']['legend_panel_height'] = legend_panel_height
        
    else:
        # Original vertical colorbar parameters
        target_colorbar_width = fig_width * colorbar_width_ratio
        target_colorbar_height = legend_panel_height * colorbar_height_ratio
        
        aspect = target_colorbar_width / target_colorbar_height
        shrink = colorbar_width_ratio
        
        legend_dict['color_bar_dict']['aspect'] = max(8, min(40, aspect))
        legend_dict['color_bar_dict']['shrink'] = max(0.2, min(0.95, shrink))
        legend_dict['color_bar_dict']['fraction'] = colorbar_height_ratio * 1.2
        legend_dict['color_bar_dict']['orientation'] = 'vertical'
    return plot_dict

In [112]:
def get_labels(plot_dict):
    measure = plot_dict['measure']
    if measure == 'suitability':
            plot_dict['outcome_type'] = 'suitability'
            plot_dict['outcome_label'] = "suitability"
            plot_dict['full_outcome_label'] = "suitability (days per year)"
    else:
        metric = plot_dict['metric']
        plot_dict['outcome_type'] = f"{measure}_{metric}"
        plot_dict['outcome_label'] = f"{measure} {metric}"
        plot_dict['full_outcome_label']  = f"{measure} {metric} (per 100,000 population)" if metric == 'rate' else f"{measure} {metric}"

    return plot_dict

In [113]:
def get_save_path(plot_dict):
    """Generate the save path for the outcome data."""

    base_path = plot_dict['base_path']
    Path(base_path).mkdir(parents=True, exist_ok=True)

    map_type = plot_dict['map_type']
    outcome_type = plot_dict['outcome_type']
    cause = plot_dict['cause']

    if map_type == 'outcome':
        save_path = f'{base_path}/{outcome_type}_{cause}_{map_type}_{plot_dict['period_1']['ssp_scenario']}_{plot_dict['period_1']['start_year']}.png'
    elif map_type == 'change':
        save_path = f'{base_path}/{outcome_type}_{cause}_{map_type}_{plot_dict['period_1']['ssp_scenario']}_{plot_dict['period_1']['start_year']}_{plot_dict['period_2']['start_year']}.png'
    else:
        save_path = f'{base_path}/{outcome_type}_{cause}_{map_type}_{plot_dict['period_1']['ssp_scenario']}_{plot_dict['period_1']['ssp_scenario']}_{plot_dict['period_1']['start_year']}.png'
    
    plot_dict['save_path'] = save_path
    if not plot_dict['remake_figure'] and plot_dict['save_figure'] and Path(save_path).exists():
        plot_dict['make_figure'] = False
    else:
        plot_dict['make_figure'] = True

    return plot_dict

In [114]:
def turn_off_axes(axes):
    for ax in axes:
        if ax is not None:
            ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")

In [115]:
def create_figure(plot_dict):
    layout_dict = plot_dict['layout_dict']
    fig = plt.figure(figsize=layout_dict['figsize'])
    ax_map = fig.add_axes(layout_dict['map']['coords'], projection=ccrs.PlateCarree())
    ax_legend = fig.add_axes(layout_dict['legend']['coords']) if plot_dict['have_legend_panel'] else None
    turn_off_axes([ax_map, ax_legend])
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
    return fig, ax_map, ax_legend

In [116]:
def draw_rectangles(fig, layout_dict, layout_keys, axes):
    for key in layout_keys:
        coords = layout_dict[key]['coords']
        rect = plt.Rectangle((coords[0], coords[1]), coords[2], coords[3],
                             linewidth=2, linestyle = '--', edgecolor='red', facecolor='none', zorder=100, transform=fig.transFigure)
        fig.patches.append(rect)
    for ax in axes:
        pos = ax.get_position()
        rect = Rectangle((pos.x0, pos.y0), pos.width, pos.height,
                    linewidth=1, edgecolor='blue', facecolor='none', zorder=100, transform=ax.figure.transFigure)
        ax.figure.patches.append(rect)

In [117]:
def periods_are_different(period):
    if len(period) == 1:
        return False
    else:
        period1, period2 = period
        # Normalize periods to [start_year, end_year] format
        start1 = period1[0]
        end1 = period1[0] if len(period1) == 1 else period1[1]
        
        start2 = period2[0] 
        end2 = period2[0] if len(period2) == 1 else period2[1]
        
        # Compare the actual time ranges
        return (start1, end1) != (start2, end2)

def scenarios_are_different(ssp_scenarios):
    """Check if two scenarios are different."""
    # Normalize scenarios to lower case for comparison
    if len(ssp_scenarios) != 2:
        return False  # Not enough scenarios to compare
    else:
        return ssp_scenarios[0].lower() != ssp_scenarios[1].lower()

In [118]:
def get_layout_dict(plot_dict):
    layout_dict = plot_dict['layout_dict']
    fig_width = layout_dict['fig_width']
    map_extent = plot_dict['map_dict']['map_extent']

    aspect_ratio = (map_extent[3] - map_extent[2]) / (map_extent[1] - map_extent[0])

    legend_title_height = layout_dict['legend_title_height']
    legend_panel_height = layout_dict['legend_panel_height']
    map_panel_height = fig_width * aspect_ratio
    sub_title_height = layout_dict['sub_title_height']
    title_height = layout_dict['title_height']
    
    fig_height = title_height + sub_title_height + map_panel_height + legend_panel_height + legend_title_height

    layout_dict['figsize'] = (fig_width, fig_height)
    
    panel_names = ['legend_title', 'legend', 'map', 'sub_title', 'title']
    layout_dict['panel_names'] = panel_names
    heights = [legend_title_height, legend_panel_height, map_panel_height, sub_title_height, title_height]
    height_fractions = [h / fig_height for h in heights]
    for ix, panel in enumerate(panel_names):
        panel_dict = {
            'height': heights[ix],
            'height_fraction': height_fractions[ix],
            'bottom': sum(height_fractions[:ix]),
            'text_y': sum(height_fractions[:ix]) + height_fractions[ix] / 2,
            'coords': [0, sum(height_fractions[:ix]), 1, height_fractions[ix]]
        }
        layout_dict[panel] = panel_dict
    
    keys_to_remove = ['legend_title_height', 'legend_panel_height', 'sub_title_height', 'title_height']
    if not plot_dict['have_legend_panel']:
        keys_to_remove.append('legend')
    for key in keys_to_remove:
        del layout_dict[key]
    return plot_dict

In [119]:
def get_period_info(plot_dict):
    map_type = plot_dict['map_type']
    ssp_scenarios = plot_dict['ssp_scenarios']
    periods = plot_dict['periods']
    # Initialize period_dict
    if map_type == 'change':
        if len(periods) != 2 or scenarios_are_different(ssp_scenarios):
            raise ValueError("Invalid temporal comparison. Need exactly two periods, 0 or 2 period labels and one SSP scenario.")
        else:
            # Create period configurations for temporal comparison
            for i, period in enumerate(periods, 1):
                if len(period) == 1:
                    start_year = end_year = period[0]
                else:
                    start_year, end_year = period
                
                plot_dict[f'period_{i}'] = {
                    'start_year': start_year,
                    'end_year': end_year,
                    'ssp_scenario': ssp_scenarios[0]  # Same scenario for both periods
                }
       
    elif map_type == 'scenario_comparison':
        if len(ssp_scenarios) != 2 or periods_are_different(periods):
            raise ValueError("Invalid scenario comparison. Need exactly two SSP scenarios and one period / period label.")
        else:
            # Create period configurations for scenario comparison (same period, different scenarios)
            period_years = periods[0]
            if len(period_years) == 1:
                start_year = end_year = period_years[0]
            else:
                start_year, end_year = period_years
            
            for i, ssp_scenario in enumerate(ssp_scenarios, 1):
                plot_dict[f'period_{i}'] = {
                    'start_year': start_year,
                    'end_year': end_year,
                    'ssp_scenario': ssp_scenario
                }            
    elif map_type == 'outcome':
        if len(periods) != 1 or len(ssp_scenarios) != 1:
            raise ValueError("For impact evaluation, need exactly one period and one SSP scenario.")
        else:
            # Create single period configuration
            period_years = periods[0]
            if len(period_years) == 1:
                start_year = end_year = period_years[0]
            else:
                start_year, end_year = period_years
            
            plot_dict['period_1'] = {
                'start_year': start_year,
                'end_year': end_year,
                'ssp_scenario': ssp_scenarios[0]
            }
            
    elif map_type == 'arbitrary_comparison':
        if len(periods) != 2 or len(ssp_scenarios) != 2:
            raise ValueError("For arbitrary comparison, need exactly two periods and two SSP scenarios.")
        else:
            # Create period configurations for arbitrary comparison
            for i, (period, ssp_scenario) in enumerate(zip(periods, ssp_scenarios), 1):
                if len(period) == 1:
                    start_year = end_year = period[0]
                else:
                    start_year, end_year = period   
                plot_dict[f'period_{i}'] = {
                    'start_year': start_year,
                    'end_year': end_year,
                    'ssp_scenario': ssp_scenario
                }
            
    else:
        raise ValueError("Invalid plot type. Choose from 'change', 'scenario_comparison', 'outcome', or 'arbitrary_comparison'.")    
    return plot_dict

In [120]:
def create_plot_dict(cause, measure, period_1, ssp_scenarios = ['ssp245'], period_2=None,
                    metric = None, 
                    # Core parameters
                    resolution='0.1', 
                    statistic='mean',
                    map_type='change',
                    per_capita=False,
                    data_type='raster',
                    location_type='endemic', # 'endemic' or 'all'
                    have_legend_panel=True,
                    base_path=None,
                    save_figure=True,
                    remake_figure=False,
                    return_figure=False,
                    # Map types:
                    # change: period 1 != period 2; scenario 1 == scenario 2
                    # scenario_comparison: period 1 == period 2; scenario 1 != scenario 2
                    # outcome: period 1 == period 2; scenario 1 == scenario 2
                    # arbitrary_comparison: period 1 != period 2; scenario 1 != scenario 2
                     
                    # Figure information
                    fig_width=12, title_height=0.5, sub_title_height=0, 
                    legend_panel_height=0.75, legend_title_height=0.25,
                    fig_height=8, linewidth=0.05,
                     
                    lat_lon_font_size= 18, inset_label_font_size= 14,
                    tick_font_size=14, water_color='#A6B6DC',
                    water_alpha=0.5,
                    run_date=run_date,
                     # Color infromtation

                     # Map information

                     # Legend infromation
                     use_colorbar=False,

                     # Map extent
                     lat_min=-60, lat_max=90, lon_min=-180, lon_max=180,
                     lat_zoom_min = -55, lat_zoom_max = 50,
                     
                     # Titles and labels
                     title=None, subtitle=None, subtitle3=None,
                     period_labels=None,
                     
                     # Plot styling
                     num_categories=9, custom_bins=None,
                     add_stats=False,
                     
                     # Colors
                     base_cmap=None,
                     masked_color='#f0f0f0', masked_alpha=1.0,
                     
                     # Font sizes
                     title_fontsize=22, legend_title_fontsize=18,
                     legend_label_fontsize=14, stats_fontsize=12,
                     
                     # Layout
                     colorbar_height=0.05, colorbar_pad=0.08, legend_bin_spacing=0.01,
                     legend_margin=0.05, legend_spacing_factor=1.0, 
                     bin_bottom =0.425, bin_top=0.85, bin_label_gap=0.075):

    if map_type == 'outcome':
        periods = [period_1]
    else:
        periods = [period_1, period_2]


    if map_type == 'outcome':
        if measure == 'suitability':
            base_cmap = 'BuPu_r'
        elif measure == 'floods':
            base_cmap = 'viridis_r'
        else:
            base_cmap = 'YlOrRd_r'
    else:
        if measure == 'suitability':
            base_cmap = 'PRGn_r'
        elif measure == 'floods':
            base_cmap = 'viridis_r'
        else:
            base_cmap = 'RdYlBu_r'

    if base_cmap is None:
        base_cmap = 'RdBu_r' if map_type in ['change', 'scenario_comparison', 'arbitrary_comparison'] else 'Reds'

    # Use generated subtitle if none provided
    if data_type == 'admin2':
        map_extent = [lon_min, lon_max, lat_zoom_min, lat_zoom_max]
    else:
        map_extent = [lon_min, lon_max, lat_min, lat_max]

    # Build the plot dictionary
    plot_dict = {
        'cause': cause,
        'measure': measure,
        'metric': metric,
        'units': 'people-days' if measure == 'floods' else 'people-hours',
        'map_type': map_type,
        'per_capita': per_capita,
        'data_type': data_type,
        'location_type': location_type,
        'periods': periods,
        'ssp_scenarios': ssp_scenarios,
        'resolution': resolution,
        'statistic': statistic,
        'title': title,
        'subtitle': subtitle,
        'subtitle3': subtitle3,
        'num_categories': num_categories,
        'custom_bins': custom_bins,
        'add_stats': add_stats,
        'have_legend_panel': have_legend_panel,
        'base_path': base_path,
        'save_figure': save_figure,
        'remake_figure': remake_figure,
        'return_figure': return_figure,
        'layout_dict': {
            'fig_width': fig_width,
            'title_height': title_height,
            'sub_title_height': sub_title_height,
            'legend_panel_height': legend_panel_height,
            'legend_title_height': legend_title_height,
        },
        'figure_dict':{
            'linewidth':linewidth,
            'lat_lon_font_size': lat_lon_font_size,
            'inset_label_font_size': inset_label_font_size,
            'tick_font_size': tick_font_size,
            'water_color':water_color,
            'water_alpha':water_alpha,
        },
        'colors_dict': {
            'base_cmap': base_cmap,
            'water_color': water_color,
            'water_alpha': water_alpha,
            'masked_color': masked_color,
            'masked_alpha': masked_alpha,
        },
        'map_dict': {    
            'admin0_polygons': admin0_polygons,
            'admin1_polygons': admin1_polygons,
            'admin2_polygons': admin2_polygons,
            'plot_admin0s': True,
            'map_extent': map_extent,
            'raster_extent': [-180, 180, -90, 90]
        },
        'legend_dict': {
            'use_colorbar': use_colorbar,
            'legend_title': None,
            'legend_panel': {
                'legend_bin_spacing': legend_bin_spacing,
                'legend_margin': legend_margin,
                'legend_spacing_factor': legend_spacing_factor,
                'bin_bottom': bin_bottom,
                'bin_top': bin_top,
                'bin_label_gap': bin_label_gap
            },
            'color_bar_dict': {
                'colorbar_height': colorbar_height,
                'colorbar_pad': colorbar_pad,
                'colorbar_width_ratio': 0.7,
                'colorbar_height_ratio': 0.5,
                'shrink': 0.9,
                'pad': 0.15,
                'aspect': 40, 
                'fraction':0.05
            }
        },
        'fontsizes': {
            'title_fontsize': title_fontsize,
            'legend_label_fontsize': legend_label_fontsize,
            'legend_title_fontsize': legend_title_fontsize,
            'stats_fontsize': stats_fontsize,
        }
    }
    plot_dict = get_layout_dict(plot_dict)
    plot_dict = get_period_info(plot_dict)
    plot_dict['outcome_type'] = f"{measure}_{metric}" if metric else 'suitability'
    plot_dict['outcome_label'] = f"{measure} {metric}" if metric else 'suitability'
    plot_dict['full_outcome_label'] = f"{measure} {metric}" if metric else "Suitability (days per year)"
    plot_dict['legend_dict']['legend_title'] = plot_dict['full_outcome_label']
    bin_key = (None, measure, None, map_type) if measure == 'suitability' else (cause, measure, metric, map_type)
    bins = bins_dictionary[bin_key]
    n_bins = len(bins) - 1
    plot_dict['bin_dict']={
        'bins':bins,
        'n_bins': n_bins,
    }
    
    return plot_dict

In [121]:
def save_figure(fig, save_path, dpi=720, facecolor='white', edgecolor='none'):
    fig.savefig(save_path, dpi=dpi,
                facecolor=facecolor, edgecolor=edgecolor)
    print(f"Figure saved to {save_path}")

In [122]:
def plot_map(plot_dict):

    plot_dict = get_save_path(plot_dict)

    if not plot_dict['make_figure']:
        print(f"Figure already exists: {plot_dict['save_path']}")
        return
    # else:
    #     print(f"Creating figure {plot_dict['save_path']}")

    plot_dict = get_labels(plot_dict)
    plot_dict = get_plot_data(plot_dict)
    # plot_dict = calculate_colorbar_params(plot_dict)

    fig, ax_map, ax_legend = create_figure(plot_dict)
    ax_map = setup_map_plot(ax_map, plot_dict)

    if plot_dict['data_type'] == 'raster':
        add_map(ax_map, plot_dict)
    else:
        plot_base_admins(ax_map, plot_dict)
        plot_data_admins(ax_map, plot_dict)
    
    figure_dict = plot_dict['figure_dict']
    layout_dict = plot_dict['layout_dict']

    # Set title and labels
    fig.text(0.5, layout_dict['title']['text_y'], figure_dict['title'], ha='center', va='center',
             fontsize=plot_dict['fontsizes']['title_fontsize'])
    #  ax_map.set_title(figure_dict['title'], fontsize=plot_dict['fontsizes']['title_fontsize'])
    # ax_map.set_xlabel("Longitude", fontsize=figure_dict['lat_lon_font_size'])
    # ax_map.set_ylabel("Latitude", fontsize=figure_dict['lat_lon_font_size'])
    
    # Add colorbar and legend
    if plot_dict['have_legend_panel']:
        add_legend(fig, ax_legend, plot_dict)
        if plot_dict['legend_dict']['legend_title'] is not None:
            fig.text(0.5, layout_dict['legend_title']['text_y'], plot_dict['legend_dict']['legend_title'], ha='center', va='center', 
                fontsize=plot_dict['fontsizes']['legend_title_fontsize'])
    else:
        add_legend(fig, ax_map, plot_dict)
    if figure_dict['inset_label'] is not None:
        add_inset(ax_map, figure_dict)

    # draw_rectangles(fig, layout_dict, layout_dict['panel_names'], [ax_map, ax_legend])
    if plot_dict['save_figure']:    
        if plot_dict['save_path'] is not None:
            save_figure(fig, plot_dict['save_path'])
        else:
            print("No save path provided or generated, figure not saved.")
    
    if plot_dict['return_fig']:
        return plot_dict, fig
    else:
        plt.close(fig)  # Add this line to prevent display
        return None, None

In [ ]:
measure = 'suitability' # 'suitability', 'cases', 'deaths', 'dalys', 'floods'
metric = None   # 'count', 'rate', 'pop_at_risk' or None for suitability
data_type = 'admin2'
location_type = 'all'
ssp_scenarios = ['ssp245']#, 'ssp585']    
remake_figure = True 
for cause in ['malaria', 'dengue']:
    for map_type in ['outcome', 'change']:
        if map_type == 'outcome':
            for year_1 in [2025,2050,2100]:
                period_1 = [year_1]
                period_2 = None # [2020, 2025] or [2025]                
                plot_dict = create_plot_dict(
                    cause=cause,
                    measure=measure,
                    period_1=period_1, 
                    period_2=period_2,
                    ssp_scenarios=ssp_scenarios,
                    map_type=map_type,
                    location_type=location_type,
                    data_type=data_type,
                    remake_figure=remake_figure,
                    base_path=f'{VISUALIZATION_PATH}/{run_date}'
                )
                plot_dict, fig = plot_map(plot_dict)
        else:
            for year_2 in [2050,2100]:
                period_1=[2025]
                period_2 = [year_2]
                plot_dict = create_plot_dict(
                    cause=cause,
                    measure=measure,
                    period_1=period_1,
                    period_2=period_2,
                    ssp_scenarios=ssp_scenarios,
                    map_type=map_type,
                    location_type=location_type,
                    data_type=data_type,
                    remake_figure=remake_figure,
                    base_path=f'{VISUALIZATION_PATH}/{run_date}'
                )
                plot_dict, fig = plot_map(plot_dict)